In [ ]:
%matplotlib inline
from pathlib import Path
import sys
import shutil
import math
import os
import json
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm, trange
import torchmetrics
import matplotlib.pyplot as plt
from pprint import pprint
import utils as course_utils

# ======== Your controls: change these to run the notebook your way ========
# True: a small run on a tiny generated dataset; grading always uses it.
# False: training on real MNIST (downloaded to ~/ once, about 60 MB).
IS_UNITTEST = True
# Epochs of each of the two networks when IS_UNITTEST = False (about
# 30 seconds per network for 150 epochs on a laptop CPU).
EPOCHS = 150
# Device of the full run. 'auto' (the default) takes the first available of
# 'cuda' (NVIDIA GPU), 'xla' (TPU), 'mps' (Apple GPU, only when Python runs on
# macOS itself) and 'cpu'; the cell prints the one it took. The small run
# always uses the CPU.
DEVICE = 'auto'
# ==========================================================================
# Grading always uses the small run, whatever IS_UNITTEST is set to above.
IS_UNITTEST = IS_UNITTEST or course_utils.checker.is_unittest_run()
# Thread count changes the order of float sums; the small run must
# repeat the teacher's numbers on any machine.
if IS_UNITTEST:
    torch.set_num_threads(1)

# -- start --
data_dir = course_utils.checker.get_data_dir()
course_utils.checker.save_npz({'seed': np.array(0)}, data_dir / 'inputs_public.npz')
course_utils.checker.save_npz({'seed': np.array(42)}, data_dir / 'inputs_private.npz')
if not course_utils.checker.is_valid_npz('xinputs.npz'):
    shutil.copyfile(data_dir / 'inputs_public.npz', 'xinputs.npz')
# -- end --

seed = int(course_utils.checker.load_answer('xinputs.npz')['seed'])

import source
# -- start --
# The teacher's notebook runs with the teacher's copy of the shared code.
if course_utils.hidden.detect():
    course_utils.hidden.replace_code()
# -- end --
src = source.code.src
answer = {}

# The device of the training: always the CPU in the small run, DEVICE otherwise.
device = src.utils.device.select_device('cpu' if IS_UNITTEST else DEVICE)
print(f"Training device: {src.utils.device.describe(device)}"
      + (" (the small run always uses the CPU)" if IS_UNITTEST else f" (DEVICE = {DEVICE!r})"))


# Dropout: Training and Evaluation Modes

A network with more weights than training examples can learn the training
set by heart: the training loss goes to zero while the validation loss
grows. This is **overfitting**.

**Dropout** fights it by noise. During training, every signal after a
layer is set to zero with probability $p$, a new random choice in every
step, so no neuron can rely on one particular other neuron being there.
At evaluation all neurons work; the network acts like an average of the
many thinned networks it trained.

For the evaluation to see the same signal size as the training did, the
kept signals are divided by $1 - p$ during training (*inverted dropout*):

$$\text{dropout}(x) = \begin{cases} \dfrac{m \odot x}{1 - p}, & m_i \sim \mathrm{Bernoulli}(1 - p) & \text{training} \\[2mm] x & & \text{evaluation} \end{cases}$$

so $\mathbb{E}[\text{dropout}(x)] = x$ in both modes. A module knows its
mode from `self.training`: `model.train()` sets it to `True` in all
submodules, `model.eval()` to `False`. `train_model` already calls
`model.train()` before every training step and `model.eval()` before
validation.

**Files to edit:** [src/models/feedforward/dropout.py](../../source/code/src/models/feedforward/dropout.py) and
the marked cell of this notebook.


In [ ]:
src.utils.seed.seed_all(0)


# Task 1: The Dropout Module

## Goal

Implement `forward` of `Dropout` in [src/models/feedforward/dropout.py](../../source/code/src/models/feedforward/dropout.py):

- in training mode with `p > 0`: a mask of the shape of the signal,
  `torch.rand(signal.shape, generator=self.generator) >= self.p`, then
  `signal * mask / (1 - p)`;
- in evaluation mode: the signal unchanged.


In [ ]:
check_signal = torch.arange(1.0, 13.0).reshape(3, 4)

# Edit: ../../source/code/src/models/feedforward/dropout.py
dropout = src.models.feedforward.dropout.Dropout(p=0.3, generator=torch.Generator().manual_seed(seed))
# ^^^ This class should be implemented in this task

dropout.train()
training_output = dropout(check_signal)
dropout.eval()
evaluation_output = dropout(check_signal)
print('training mode:', training_output, 'evaluation mode:', evaluation_output, sep='\n')

dropout.train()
ones = dropout(torch.ones(100000))
answer['dropout'] = {
    'training': training_output,
    'evaluation': evaluation_output,
    'zero_share': (ones == 0).float().mean(),
    'mean': ones.mean(),
}
print(f"share of zeros {float(answer['dropout']['zero_share']):.3f}, mean {float(answer['dropout']['mean']):.3f}")


In [ ]:
course_utils.checker.print_progress(
    'Task 1: The Dropout Module', answer,
    prefixes=('dropout',),
    is_small_run=IS_UNITTEST)


# Task 2: A Network with Dropout

## Goal

`make_network(p, p_input)` builds a `SequentialClassifier` (given in the
same file: it flattens the images and applies a list of layers) of
784 → 512 → 512 → 10 neurons with ReLU, with dropout in two places, as in
the original dropout paper:

- `Dropout(p_input)` on the input pixels, before the first layer;
- `Dropout(p)` after every hidden ReLU.

With `p = p_input = 0` it is the same network without dropout.

The data: only the first 1 000 images of the MNIST training set, few
enough for the network to learn them by heart.


In [ ]:
MNIST_train = src.datasets.mnist_simple.MNISTSimpleDataset(train=True, small=IS_UNITTEST, seed=seed)
MNIST_valid = src.datasets.mnist_simple.MNISTSimpleDataset(train=False, small=IS_UNITTEST, seed=seed)
train_subset = torch.utils.data.Subset(MNIST_train, range(min(len(MNIST_train), 256 if IS_UNITTEST else 1000)))
print('training images:', len(train_subset), 'validation images:', len(MNIST_valid))
Dropout = src.models.feedforward.dropout.Dropout
SequentialClassifier = src.models.feedforward.dropout.SequentialClassifier

# Edit this cell in task.ipynb.
def make_network(p, p_input):
    return SequentialClassifier([
        torch.nn.Linear(28 * 28, 512), torch.nn.ReLU(),
        torch.nn.Linear(512, 512), torch.nn.ReLU(),
        torch.nn.Linear(512, 10)])

## YOUR CODE HERE
# -- start --
def make_network(p, p_input):
    return SequentialClassifier([
        Dropout(p_input),
        torch.nn.Linear(28 * 28, 512), torch.nn.ReLU(), Dropout(p),
        torch.nn.Linear(512, 512), torch.nn.ReLU(), Dropout(p),
        torch.nn.Linear(512, 10)])
# -- end --

check_network = make_network(0.5, 0.2)
print(check_network)
src.utils.deterministic_init(check_network)
check_network.eval()
answer['network'] = {
    'n_dropouts': sum(isinstance(module, Dropout) for module in check_network.modules()),
    'evaluation_output': check_network({'data': {'image': torch.linspace(-1, 1, 2 * 784).reshape(2, 28, 28)}})['signals']['output'],
}


In [ ]:
course_utils.checker.print_progress(
    'Task 2: A Network with Dropout', answer,
    prefixes=('network',),
    is_small_run=IS_UNITTEST)


# The Experiment

The same network without dropout and with dropout ($p = 0.5$ after the
hidden layers, $p_\text{input} = 0.2$ on the pixels) trains on the
1 000 images with `Adam` (`lr=1e-3`), batches of 100, for `EPOCHS` epochs
(3 in the small run); validation uses all 10 000 test images.


In [ ]:
def loss(batch):
    return F.cross_entropy(batch['signals']['output'], batch['data']['label'])


def accuracy(batch):
    return (batch['postprocessed']['class'] == batch['data']['label']).float().sum(), len(batch['data']['label'])


train_dl = torch.utils.data.DataLoader(
    train_subset, batch_size=100 if not IS_UNITTEST else 32, shuffle=True,
    generator=torch.Generator().manual_seed(0))
valid_dl = torch.utils.data.DataLoader(MNIST_valid, batch_size=1000, shuffle=False)

n_epochs = 3 if IS_UNITTEST else EPOCHS
histories = {}
for p, p_input in ((0.0, 0.0), (0.5, 0.2)):
    src.utils.seed.seed_all(0)
    network = make_network(p, p_input)
    optimizer = torch.optim.Adam(network.parameters(), lr=1.0e-3)
    history = src.train_loop.train_model(
        network, n_epochs, train_dl, valid_dl, {'ce': loss}, optimizer,
        metrics={'accuracy': accuracy}, run_name=f'p={p}',
        device=device)
    histories['no dropout' if p == 0 else 'dropout'] = {
        'train_loss': history['train_loss']['ce'],
        'valid_loss': history['valid_loss']['ce'],
        'valid_accuracy': history['valid_metrics']['accuracy'],
    }
answer['experiments'] = histories

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for name, history in histories.items():
    epochs = range(1, len(history['valid_loss']) + 1)
    line, = axes[0].plot(epochs, history['valid_loss'], label=f'{name}, validation')
    axes[0].plot(epochs, history['train_loss'], '--', color=line.get_color(), label=f'{name}, training')
    axes[1].plot(epochs, history['valid_accuracy'], label=name)
axes[0].set_yscale('log')
axes[0].set_title('loss')
axes[1].set_title('validation accuracy')
for ax in axes:
    ax.set_xlabel('epoch')
    ax.legend()
plt.show()
for name, history in histories.items():
    best = int(np.argmin(history['valid_loss']))
    print(f"{name}: final validation loss {history['valid_loss'][-1]:.3f} (lowest {history['valid_loss'][best]:.3f}"
          f" at epoch {best + 1}), final accuracy {history['valid_accuracy'][-1]:.3f}")


In [ ]:
course_utils.checker.print_progress(
    'The Experiment', answer,
    prefixes=('experiments',),
    is_small_run=IS_UNITTEST)


What to look for (full run, 150 epochs, about a minute):

- both networks learn the 1 000 images by heart: the training loss
  falls to $10^{-3}$ and below;
- **without dropout** the validation loss is lowest after a few epochs and
  then grows (to about 0.76): the network becomes more and more sure of its
  answers, including the wrong ones;
- **with dropout** the validation loss stays much lower (about 0.50) and
  the accuracy is about 2% higher (about 0.90 against 0.88). Dropout does
  not turn 1 000 images into 60 000: it helps, but the large gain comes
  from more data;
- the training loss with dropout is measured in training mode, with
  neurons switched off, so it is higher than that of the same network in
  evaluation mode.


In [ ]:
course_utils.checker.save_answer(answer)
